# GENIE generator comparisons (unfolded data vs MC truth vs GENIE flat ROOT)

Compare **unfolded beam data** differential cross sections to:

1. **MC truth (production `mcnu`)** — the curve labeled **GENIE** in `unfolding-data.ipynb`: `nevts_allmc × xsec_unit` from weighted `mcnu` (same POT scale as data).
2. **GENIE standalone flat ROOT** — truth-level generator prediction from `FlatTree_VARS` (same 1p0pi topology selection as `generator_comparison.ipynb`).

All curves are plotted in **absolute differential cross section** units (`dσ/dX` per `VariableConfig.xsec_label`), **not** shape- or density-normalized.

### Normalization (read carefully)

| Source | Event weight / count | Cross section |
|--------|----------------------|---------------|
| **Unfolded data** | Wiener `unfold['unfold']` | `σ_bin / Δbin` |
| **MC truth (`mcnu`)** | `nevts_allmc` with `pot_weight = data_tot_pot / mc_tot_pot` | `nevts_allmc × xsec_unit / Δbin` |
| **GENIE flat ROOT** | `40 × fScaleFactor × Weight` per selected event | histogram sum `× (data_tot_pot / GENIE_REF_POT) / Δbin` |

Shared exposure factor (from `unfolding-data.ipynb`):

`xsec_unit = 1 / ( ∫Φ dE × POT × N_targets )` with ray-traced **FV_split_truncY**, **Gen1.root** flux, and **data_tot_pot**.

Standalone GENIE flat files are generated for **`GENIE_REF_POT = 1×10²⁰`** (config cell). Exposure is in **`40 × fScaleFactor × Weight`** per event; **`fScaleFactor` is not `1/POT`** (do not invert it). Curves are scaled to beam POT via `data_tot_pot / GENIE_REF_POT`, analogous to `pot_weight` on `mcnu`.

### Signal definition (kinematics & FV)

| Stage | Selection |
|-------|-----------|
| **Reco data / MC evt** | `perTPC_cut` on vertex + μ/p track ends (10 cm cathode inset) |
| **Unfold Wiener model** | `signal_hists(..., mode='unfold', signal_truth_fv='none')` → `IsNuInFV_NumuCC_1p0pi` with **`signal_truth_fv='none'`** (topology + vertex FV, no μ/p end containment) |
| **MC truth (plots)** | `MC_TRUTH_SIGNAL_MODE='unfold'`: `nevts_allmc × xsec_unit` from `signal_hists` (`IsNuInFV_NumuCC_1p0pi`, no containment) |
| **GENIE flat ROOT** | `40 × fScaleFactor × Weight`, scaled to beam POT; **topology only** (νμ CC + 1μ1p0π with highest-$p$ leading μ/p; **no vertex FV** on flat files). GiBUU: extra factor **1000**. |

Set **`SIGNAL_TRUTH_FV`** for unfold / MC truth. The GENIE flat **1p0π signal mask** is defined explicitly in §Load GENIE flat (no vertex FV).


In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
from os import path, makedirs
from datetime import datetime
from functools import partial
from pathlib import Path

import numpy as np
import pandas as pd
import uproot

import sys
sys.path.append('/exp/sbnd/app/users/munjung/xsec/freeze/cafpyana')
from pyanalib.split_df_helpers import load_dfs
from pyanalib.split_df_helpers_new import dfs_from_dir
from analysis_village.numucc_1p0pi.variable_configs import VariableConfig
from analysis_village.numucc_1p0pi.utils import *
from analysis_village.numucc_1p0pi.categories import (
    IsNuInFV_NumuCC_1p0pi,
    PER_TPC_INCATHODE_CM,
    DETECTOR as SIGNAL_DETECTOR,
)
from analysis_village.numucc_1p0pi.files_config import save_fig_base_dir
from analysis_village.numucc_1p0pi.constants import *
from analysis_village.numucc_1p0pi.dataset_locations import PLOTS_BASE, GENIE_GROUP_GLOBS
from analysis_village.numucc_1p0pi.syst_disk_layout import category_summary_npz_path
from analysis_village.unfolding.wienersvd import *
from pyanalib.covariance import *

import matplotlib.pyplot as plt

import warnings
from pandas.errors import PerformanceWarning
warnings.filterwarnings("ignore", category=PerformanceWarning)
plt.style.use("presentation.mplstyle")

In [3]:
save_fig = True
today_str = datetime.now().strftime("%Y%m%d")
today_str = "thesis"
save_fig_dir = path.join(save_fig_base_dir, f"genie_generator_comparisons-{today_str}")
if save_fig and not path.exists(save_fig_dir):
    makedirs(save_fig_dir)
print("saving plots in", save_fig_dir)

approval = ""

DFS_ROOT = "/pnfs/sbnd/scratch/users/munjung/cafpyana_out/dfs"
QUALITY_DF_PATH = path.join(DFS_ROOT, "2026_05_16_230705__sel_mup-data-1e20/merged_perTPC/beam_data_1e20_qualitycut.df")
DIR_GENIE_CCQE = path.dirname(GENIE_GROUP_GLOBS["CCQE"])

SYST_DISK_ROOT = "/exp/sbnd/data/users/munjung/plots/numucc1p0pi/systematics-final"
CATEGORY_SUMMARY_OUT = Path(PLOTS_BASE) / "syst_uncertainty_breakdown" / "final_selected" / "category_syst_summary.npz"
CATEGORY_SUMMARY_NPZ = category_summary_npz_path(SYST_DISK_ROOT)
if not path.isfile(CATEGORY_SUMMARY_NPZ) and CATEGORY_SUMMARY_OUT.is_file():
    CATEGORY_SUMMARY_NPZ = str(CATEGORY_SUMMARY_OUT)
UNFOLDING_SYST_KIND = "xsec"

# Truth FV for unfold (reco/per-TPC)
SIGNAL_TRUTH_FV = "per_tpc"  # or "nominal"
MC_TRUTH_SIGNAL_MODE = "unfold"  # production mcnu (signal_hists)
# MC vtx FV for integrated closure checks only (GENIE flat: no vertex cut — see §Load GENIE flat)
FLAT_ALIGNED_X_CM = 10.0
GENIE_FLAT_FV_DET = "SBND_nohighyz"

# Reference POT in GENIE flat files (exposure is in 40×fScaleFactor×Weight, not 1/fScaleFactor)
GENIE_REF_POT = 1.0e20

# GENIE standalone flat ROOT (same family as genie_vs_production_xsec.ipynb)
GENIE_FLAT_DIR = "/pnfs/sbnd/persistent/users/apapadop/GENIETweakedSamples/v3_6_2_AR23_20i_00_000_gen1_flux"
GENIE_FLAT_FILES = {
    "GENIE AR23 (CC)": GENIE_FLAT_DIR + "/14_1000180400_CC_v3_6_2_AR23_20i_00_000.flat.root",
    # "GENIE AR23 (CCQE)": GENIE_FLAT_DIR + "/14_1000180400_CCQE_v3_6_2_AR23_20i_00_000.flat.root",
}
# Optional: event-level vtx (m) keyed by flat-file label — see flux_closure/make_df4xsec.py
# Optional pkl sidecar (ar23.pkl is a small subsample only — full flat files use px_vert branches)
GENIE_VTX_PKL = {
    # "GENIE AR23 (CC)": "/exp/sbnd/data/users/munjung/xsec/flux_closure/ar23.pkl",
}

KEYS2LOAD_MC = ["hdr", "evt", "mcnu"]
N_MAX_CONCAT = 999
C_type = 2
Norm_type = 0.0
breakdown_type = "topology"


saving plots in /exp/sbnd/data/users/munjung/plots/numucc1p0pi/genie_generator_comparisons-thesis


## Load beam data and production MC (`mcnu`)

In [4]:
# quality_dfs = load_dfs(
#     QUALITY_DF_PATH,
#     keys2load=["hdr", "trigger", "evt_good"],
#     n_max_concat=10,
# )
# data_evt_df = quality_dfs["evt_good"]
# data_hdr_df = quality_dfs["hdr"].join(quality_dfs["trigger"])
# data_evt_df[("mc", "iscc")] = 999

In [5]:
quality_dfs = load_dfs(
    QUALITY_DF_PATH,
    keys2load=["hdr", "trigger", "evt_good"],
    n_max_concat=1,
)
data_evt_df = quality_dfs["evt_good"]
data_hdr_df = quality_dfs["hdr"].join(quality_dfs["trigger"])
data_evt_df[("mc", "iscc")] = 999


genie_dfs = dfs_from_dir(
    DIR_GENIE_CCQE,
    filename_str="sel_mup-wgts_genie_CCQE",
    keys2load=KEYS2LOAD_MC,
    n_max_concat=N_MAX_CONCAT,
)
mc_evt_df = genie_dfs["evt"]
mc_hdr_df = genie_dfs["hdr"]
mc_nu_df = genie_dfs["mcnu"]


Found 19 files to process
Files to process: ['/pnfs/sbnd/scratch/users/munjung/cafpyana_out/dfs/2026_05_11_024530__sel_mup-wgts_genie_CCQE/merged_perTPC/2026_05_11_024530__sel_mup-wgts_genie_CCQE_merged_0000.df', '/pnfs/sbnd/scratch/users/munjung/cafpyana_out/dfs/2026_05_11_024530__sel_mup-wgts_genie_CCQE/merged_perTPC/2026_05_11_024530__sel_mup-wgts_genie_CCQE_merged_0002.df', '/pnfs/sbnd/scratch/users/munjung/cafpyana_out/dfs/2026_05_11_024530__sel_mup-wgts_genie_CCQE/merged_perTPC/2026_05_11_024530__sel_mup-wgts_genie_CCQE_merged_0003.df', '/pnfs/sbnd/scratch/users/munjung/cafpyana_out/dfs/2026_05_11_024530__sel_mup-wgts_genie_CCQE/merged_perTPC/2026_05_11_024530__sel_mup-wgts_genie_CCQE_merged_0004.df', '/pnfs/sbnd/scratch/users/munjung/cafpyana_out/dfs/2026_05_11_024530__sel_mup-wgts_genie_CCQE/merged_perTPC/2026_05_11_024530__sel_mup-wgts_genie_CCQE_merged_0005.df', '/pnfs/sbnd/scratch/users/munjung/cafpyana_out/dfs/2026_05_11_024530__sel_mup-wgts_genie_CCQE/merged_perTPC/2026_05

100%|██████████| 19/19 [02:15<00:00,  7.14s/it]


REMEMBER TO RECALCULATE TKI AND CHECK FV!!


In [6]:
data_tot_pot = data_hdr_df["pot"].sum() * 0.9822
# data_tot_pot = 8.818e+19
pot_str = get_pot_str(data_tot_pot)
pot_label = f"Events / Bin (POT={pot_str})"
data_evt_df["pot_weight"] = np.ones(len(data_evt_df))
print("data_tot_pot: %.3e" % data_tot_pot)

mc_tot_pot = mc_hdr_df["pot"].sum()
mc_pot_scale = data_tot_pot / mc_tot_pot
mc_evt_df["pot_weight"] = mc_pot_scale * np.ones(len(mc_evt_df))
mc_nu_df["pot_weight"] = mc_pot_scale * np.ones(len(mc_nu_df))
print(f"MC: evt={len(mc_evt_df):,}  mcnu={len(mc_nu_df):,}  mc_tot_pot={mc_tot_pot:.3e}  mc_pot_scale={mc_pot_scale:.3e}")

data_tot_pot: 8.808e+19
MC: evt=70,535  mcnu=10,933,450  mc_tot_pot=5.347e+20  mc_pot_scale=1.647e-01


In [7]:
DIR_OFFBEAM = path.join(DFS_ROOT, "2026_05_11_040015__sel_mup-data-OffBeamLight/merged_perTPC")

KEYS2LOAD = ["hdr", "evt"]

offbeam_dfs = dfs_from_dir(
    DIR_OFFBEAM,
    filename_str="sel_mup-data-OffBeamLight",
    keys2load=KEYS2LOAD,
    n_max_concat=999,
)

offbeam_evt_df = offbeam_dfs["evt"]
offbeam_hdr_df = offbeam_dfs["hdr"]

offbeam_evt_df.loc[offbeam_evt_df.mc.iscc.isna(), ("mc", "iscc")] = 999

Found 1 files to process
Files to process: ['/pnfs/sbnd/scratch/users/munjung/cafpyana_out/dfs/2026_05_11_040015__sel_mup-data-OffBeamLight/merged_perTPC/2026_05_11_040015__sel_mup-data-OffBeamLight_merged_0000.df']


100%|██████████| 1/1 [00:02<00:00,  2.49s/it]

REMEMBER TO RECALCULATE TKI AND CHECK FV!!


In [8]:
data_gates = data_hdr_df.nbnbinfo.sum()
data_evt_df["pot_weight"] = np.ones(len(data_evt_df))

mc_tot_pot = mc_hdr_df["pot"].sum()
mc_pot_scale = data_tot_pot / mc_tot_pot
print(f"mc_pot_scale: {mc_pot_scale:.3e}")
mc_evt_df["pot_weight"] = mc_pot_scale * np.ones(len(mc_evt_df))

offbeam_gates = offbeam_hdr_df[offbeam_hdr_df["first_in_subrun"] == 1]["noffbeambnb"].sum()
f = 0.08
scale_offbeam_to_lightdata = (1 - f) * data_gates / offbeam_gates
print(f"offbeam data scale: {scale_offbeam_to_lightdata:.2f}")
offbeam_evt_df["pot_weight"] = scale_offbeam_to_lightdata * np.ones(len(offbeam_evt_df))

mc_pot_scale: 1.647e-01
offbeam data scale: 0.22


In [9]:
perTPC_inset = 10

def perTPC_cut(df):
    in_TPC1_cut = (
        InFV(df.slc.vertex, det="SBND_TPC1", incathode=perTPC_inset)
        & InFV(df.mu.pfp.trk.end, det="SBND_TPC1", incathode=perTPC_inset)
        & InFV(df.p.pfp.trk.end, det="SBND_TPC1", incathode=perTPC_inset)
    )
    in_TPC2_cut = (
        InFV(df.slc.vertex, det="SBND_TPC2", incathode=perTPC_inset)
        & InFV(df.mu.pfp.trk.end, det="SBND_TPC2", incathode=perTPC_inset)
        & InFV(df.p.pfp.trk.end, det="SBND_TPC2", incathode=perTPC_inset)
    )
    return in_TPC1_cut | in_TPC2_cut

data_evt_df = data_evt_df[perTPC_cut(data_evt_df)]
mc_evt_df = mc_evt_df[perTPC_cut(mc_evt_df)]
mc_evt_df.loc[mc_evt_df.mc.iscc.isna(), ("mc", "iscc")] = 999

## Exposure: `xsec_unit` (same as `unfolding-data.ipynb`)

In [10]:
from analysis_village.flux.raytrace_volume_defs import (
    FV_SPLIT_TRUNCY_BOXES,
    RAYTRACE_VOLUME_LABEL,
)

FLUX_FILE = "/exp/sbnd/data/users/munjung/flux/SBND_gsimple_raytrace/Gen1.root"
FLUX_FV = "FV_split_truncY"
XSEC_TOT_POT = data_tot_pot

print(f"Fiducial volume ({FLUX_FV}): {RAYTRACE_VOLUME_LABEL[FLUX_FV]}")
V_SBND = 0.0
for i, box in enumerate(FV_SPLIT_TRUNCY_BOXES):
    dx = box["x_range"][1] - box["x_range"][0]
    dy = box["y_range"][1] - box["y_range"][0]
    dz = box["z_range"][1] - box["z_range"][0]
    V_SBND += dx * dy * dz
print(f"V_SBND = {V_SBND:.6e} cm³")

integrated_flux_per_pot = get_integrated_flux(FLUX_FILE, plot=False)
integrated_flux = integrated_flux_per_pot * XSEC_TOT_POT
N_TARGETS = (RHO * V_SBND / M_AR) * N_A
xsec_unit = 1.0 / (integrated_flux * N_TARGETS)
print(f"integrated flux × POT = {integrated_flux:.6e} ν/cm²")
print(f"N_targets = {N_TARGETS:.4e}")
print(f"xsec_unit = {xsec_unit:.6e} cm²/nucleon")

Fiducial volume (FV_split_truncY): $10<|x|<190$: $|y|<190$ ($10<z<250$), $-190<y<100$ ($250<z<450$)
V_SBND = 5.371200e+07 cm³
Integrated flux: 1.518e-08
integrated flux × POT = 1.336994e+12 ν/cm²
N_targets = 1.1203e+30
xsec_unit = 6.676589e-43 cm²/nucleon


## Load GENIE flat ROOT (signal cut defined below)


In [11]:
import awkward as ak

from analysis_village.numucc_1p0pi.genie_flat_helpers import (
    BRANCHES_NU,
    BRANCHES_TRK,
    GIBUU_EXTRA_SCALE,
    add_genie_tki_columns,
    infer_genie_ref_pot,
    plot_generator_comparison,
    prepare_genie_trk_df,
    run_integrated_closure_checks,
    get_production_signal_mask,
)


def _genie_add_trk_counts(nudf, trkdf):
    """Match makedf mcnu / genie_vs_production_xsec: |pdg| counts above p thresholds;
    leading mu/p = highest-momentum FS particle (not first listed track)."""
    nudf = nudf.copy()
    trkdf = trkdf.copy()
    trkdf["_p"] = np.sqrt(trkdf.px**2 + trkdf.py**2 + trkdf.pz**2)
    for pid, pname, pth in zip([13, 2212, 211, 111], ["mu", "proton", "pi", "pi0"], [0.22, 0.3, 0.07, 0]):
        ntrks = trkdf[(np.abs(trkdf.pdg) == pid) & (trkdf._p > pth)].pdg.groupby(level=[0]).count()
        nudf[f"n{pname}s"] = ntrks.fillna(0)
        species = trkdf[trkdf.pdg == pid] if pname == "proton" else trkdf[np.abs(trkdf.pdg) == pid]
        leading = species.sort_values("_p", ascending=False).groupby(level=[0]).head(1)
        leading_p = leading["_p"]
        leading_p.name = f"{pname}_p"
        nudf = nudf.join(leading_p.reset_index(level=[1])[f"{pname}_p"])
        for comp, trk_col in zip(("x", "y", "z"), ("px", "py", "pz")):
            leading_dir = leading[trk_col] / leading_p
            leading_dir.name = f"{pname}_dir{comp}"
            nudf = nudf.join(leading_dir.reset_index(level=[1])[f"{pname}_dir{comp}"])
    return nudf


def _genie_numu_cc(nudf):
    return (nudf.PDGnu == 14) & (nudf.cc == 1)


def _genie_1mu_220(nudf):
    return _genie_numu_cc(nudf) & (nudf.nmus == 1)


def _genie_mu_p_lt1(nudf):
    return _genie_1mu_220(nudf) & (nudf.mu_p < 1.0)


def _genie_1p_300(nudf):
    # mc.np_300MeVc == 1: exactly one proton above 0.3 GeV; lower-p protons allowed
    return _genie_mu_p_lt1(nudf) & (nudf.nprotons == 1)


def _genie_p_p_lt1(nudf):
    # mc.p.genp < 1 GeV on highest-momentum proton (proton_p)
    return _genie_1p_300(nudf) & (nudf.proton_p < 1.0)


def _genie_no_pions(nudf):
    return (
        _genie_p_p_lt1(nudf)
        & (np.nan_to_num(nudf.npis, nan=0) == 0)
        & (np.nan_to_num(nudf.npi0s, nan=0) == 0)
    )


def genie_flat_1p0pi_signal_mask(nudf):
    """GENIE flat 1p0pi: topology only (no vertex FV). Same as genie_vs_production_xsec."""
    return _genie_no_pions(nudf)


def load_genie_flat_pack(label, flat_path, data_pot, genie_ref_pot):
    print(f"Loading {label}: {flat_path}")
    events = uproot.open(flat_path + ":FlatTree_VARS")
    nu_df = events.arrays(BRANCHES_NU, library="pd")
    trk_df = prepare_genie_trk_df(events.arrays(BRANCHES_TRK, library="ak"))
    nu_df = _genie_add_trk_counts(nu_df, trk_df)
    nu_df = add_genie_tki_columns(nu_df)
    scale_factor = nu_df.fScaleFactor.unique()[0]
    pot_scale = 1.05
    if "GiBUU" in flat_path or "gibuu" in flat_path.lower():
        pot_scale = pot_scale / GIBUU_EXTRA_SCALE
    sig_mask = genie_flat_1p0pi_signal_mask(nu_df)
    numu_cc_mask = _genie_numu_cc(nu_df)
    print(
        f"  entries={len(nu_df):,}  numuCC={int(numu_cc_mask.sum()):,}  "
        f"1p0pi signal={int(sig_mask.sum()):,}  "
        # f"GENIE_REF_POT={ref_pot:.3e}  pot_scale={pot_scale:.3e}"
    )
    return {
        "nu_df": nu_df,
        "sig_mask": sig_mask,
        "numu_cc_mask": numu_cc_mask,
        # "ref_pot": ref_pot,
        "pot_scale": pot_scale,
        "flat_path": flat_path,
    }


genie_flat_cache = {
    label: load_genie_flat_pack(label, flat_path, data_tot_pot, GENIE_REF_POT)
    for label, flat_path in GENIE_FLAT_FILES.items()
}
_mc_sig = get_production_signal_mask(mc_nu_df, SIGNAL_TRUTH_FV)
print(f"mcnu: {len(mc_nu_df):,} rows  1p0pi({SIGNAL_TRUTH_FV})={_mc_sig.sum():,}")


Loading GENIE AR23 (CC): /pnfs/sbnd/persistent/users/apapadop/GENIETweakedSamples/v3_6_2_AR23_20i_00_000_gen1_flux/14_1000180400_CC_v3_6_2_AR23_20i_00_000.flat.root
  entries=1,000,000  numuCC=1,000,000  1p0pi signal=318,037  
mcnu: 10,933,450 rows  1p0pi(per_tpc)=370,145


## Unfold beam data (same as `unfolding-data.ipynb`)


In [12]:
Norm_type = 0.

In [13]:
# chi2_list = {
# "muon-p": 9.69,
# "muon-dir_z": 27.35,
# "proton-p": 18.08,
# "proton-dir_z": 13.67,
# "tki-del_Tp": 38.8,
# "tki-del_Tp_x": 28.7,
# "tki-del_Tp_y": 26.6,
# "tki-del_p": 33.6,
# "tki-del_alpha": 3.4,
# "tki-del_phi": 9.5,
# }

In [ ]:
# get_category_summary_syst_unc is in utils (already imported via from utils import *)

def get_syst_unc(var_config, plot=False):
    return get_category_summary_syst_unc(
        var_config,
        syst_kind=UNFOLDING_SYST_KIND,
        syst_disk_root=SYST_DISK_ROOT,
        category_syst_summary_path=CATEGORY_SUMMARY_NPZ,
    )


# def get_syst_unc(var_config, plot=False):

#     outdir = "/exp/sbnd/data/users/munjung/xsec/RESULTS/DATA_RESULTS"
#     print(os.path.join(outdir, "frac_cov_dict.pkl"))
#     with open(os.path.join(outdir, "frac_cov_dict-integrated.pkl"), "rb") as f:
#         frac_cov_dict = pickle.load(f)
#     frac_cov_matrix_total = frac_cov_dict[var_config.var_save_name]

#     frac_uncert_total = np.sqrt(np.diag(frac_cov_matrix_total))
   

#     if plot:
#         plt.hist(var_config.bin_centers, bins=var_config.bins, weights=frac_uncert_total,    histtype="step", linewidth=2, color="k",  label="Total")

#         plt.xlim(var_config.bins[0], var_config.bins[-1])
#         plt.ylim(0, max(frac_uncert_total) * 1.4)

#         plt.xlabel(var_config.var_labels[1])
#         plt.ylabel("Uncertainty [%]")
#         plt.legend(fontsize=11, ncol=3, loc="upper center")

#         plt.grid(which='major', linestyle='-', linewidth=0.7, alpha=0.7)
#         plt.grid(which='minor', linestyle=':', linewidth=0.5, alpha=0.5)
#         plt.minorticks_on()

#         if not plot:
#             plt.close()
#         else:
#             plt.show();

#     return frac_uncert_total, frac_cov_matrix_total


unfolding_plotter = partial(
    overlay_hists,
    breakdown_type=breakdown_type,
    mc_df=mc_evt_df,
    data_df=data_evt_df,
    # intime_df=offbeam_evt_df,
    plot=False,
    save_fig=False,
    syst_kind=UNFOLDING_SYST_KIND,
    syst_disk_root=SYST_DISK_ROOT,
    category_syst_summary_path=CATEGORY_SUMMARY_NPZ,
    load_syst_from_summary=True,
)

var_configs = [
    # VariableConfig.all_events(),
    VariableConfig.muon_momentum(),
    VariableConfig.muon_direction(),
    VariableConfig.proton_momentum(),
    VariableConfig.proton_direction(),
    VariableConfig.tki_del_Tp(),
    VariableConfig.tki_del_Tp_x(),
    VariableConfig.tki_del_Tp_y(),
    VariableConfig.tki_del_p(),
    VariableConfig.tki_del_alpha(),
    VariableConfig.tki_del_phi(),
]

unfold_cache = {}
for var_config in var_configs:
    print("\n===", var_config.var_save_name, "===")
    syst_ret = get_syst_unc(var_config, plot=False)
    # print(syst_ret[0])
    # plot_frac_unc(syst_ret[0], var_config)
    syst_cov_matrix = syst_ret[1]
    plot_labels_hist = [var_config.var_labels[1], pot_label, ""]
    ret = unfolding_plotter(
        var_config=var_config,
        plot_labels=plot_labels_hist,
        syst=syst_cov_matrix,
    )
    ret_signal_hists = signal_hists(
        mc_evt_df, mc_nu_df, var_config, mode="unfold", return_data=True, plot=False,
        signal_truth_fv="none",
    )
    if len(var_config.bins) == 2:
        reco_vs_true = np.array([[1.0]])
    else:
        reco_vs_true, _, _ = np.histogram2d(
            ret_signal_hists["var_sel_truth"],
            ret_signal_hists["var_sel_reco"],
            weights=ret_signal_hists["wgt_sel_truth"],
            bins=var_config.bins,
        )
    eff = ret_signal_hists["nevts_sel_truth"] / ret_signal_hists["nevts_allmc"]
    response = get_response_matrix(reco_vs_true, eff)
    nevts_sel_data = ret["total_data"] - ret["total_mc_bkgd"]
    measured = nevts_sel_data * xsec_unit
    model = ret_signal_hists["nevts_allmc"] * xsec_unit
    Covariance = cov_from_fraccov(syst_cov_matrix, ret_signal_hists["nevts_sel_reco"]) * xsec_unit**2
    unfold = WienerSVD(response, model, measured, Covariance, C_type, Norm_type, stat_scaling=xsec_unit)
    unfold_cache[var_config.var_save_name] = {
        "unfold": unfold,
        "model": model,
        "measured": measured,
        "ret_signal_hists": ret_signal_hists,
        "var_config": var_config,
    }

    save_name = f"{save_fig_dir}/{var_config.var_save_name}-unfolded_event_rates-data"
    models = {"GENIE AR23_20i": [model, "C0"]} #,  "AR23": nevts_dict["GENIE AR23"]}
    plot_unfolded_result(unfold, 
                        measured, 
                        models,
                        var_config,
                        # chi2_list=chi2_list,
                        xsec_unit=xsec_unit,
                        save_fig=save_fig, 
                        save_name=save_name,
                        textloc=[0.05, 0.4],
                        approval=approval,
                        data=True,
                        closure_test=False)
    save_fig_name = "{}/{}-{}-add_smear".format(save_fig_dir, var_config.var_save_name, "data")
    plot_heatmap(
                unfold["AddSmear"],
                var_config.bins,
                plot_labels=[var_config.var_labels[2], var_config.var_labels[0], "", "$A_c$"],
                approval=approval,
                save_fig=save_fig,
                save_name=save_fig_name,
                cmap="viridis"
    )

    save_fig_name = "{}/{}-{}-syst_cov_frac".format(save_fig_dir, var_config.var_save_name, "data")
    plot_heatmap(
                syst_cov_matrix,
                var_config.bins,
                plot_labels=[var_config.var_labels[1], var_config.var_labels[1], "Covariance"],
                approval=approval,
                save_fig=save_fig,
                save_name=save_fig_name,
                cmap="viridis"
    )


=== muon-p ===
No intime cosmics provided


In [15]:
# save unfold_cache as pickle file
# outdir = "/exp/sbnd/data/users/munjung/xsec/RESULTS/DATA_RESULTS"
# with open(os.path.join(outdir, "Gen1_unfold_cache_efielddet_noneutron.pkl"), "wb") as f:
#     pickle.dump(unfold_cache, f)